# Dynamic calibration: MB parameters, ice thickness and spinup in one go

**Experimental.** This notebook tests a new workflow; we do not know yet whether
it is a good idea.

OGGM's standard workflow prepares a glacier for a simulation in three steps:

1. **MB calibration**: the MB parameters (`melt_f`, `prcp_fac`, `temp_bias`)
   are calibrated on a fixed geometry against the geodetic MB 2000-2020
   (Hugonnet et al., 2021);
2. **ice thickness inversion**: the thickness at the RGI date is computed from
   the MB (mass conservation) and the creep parameter of ice (Glen's A). A is
   calibrated for a whole region, so that the regional volume matches a
   reference (IceBoost v2 by default);
3. **dynamic calibration**: the MB calibration of step 1 uses the fixed geometry
   of the RGI date. In a dynamical run, the geometry changes and the MB with it
   (elevation feedback), so that the run no longer matches the geodetic MB.
   OGGM's dynamic calibration (`run_dynamic_melt_f_calibration`) recalibrates
   `melt_f` with dynamical runs, each of them starting with a **dynamic
   spinup**: from the inverted glacier, a short simulation with a constant
   climate and a temperature bias makes the glacier grow (or shrink), so that,
   after a run with the historical climate, the area at the RGI date matches
   the RGI outline.

Here we replace the three steps with **one** AdaPBS run for a single glacier,
with five parameters and three observations:

- parameters: the three MB parameters, Glen's A, and the temperature offset of
  the growth period of the spinup;
- observations: the geodetic MB, the RGI area and the IceBoost mean ice
  thickness.

Each particle runs the whole chain (inversion, growth, historical run), and
AdaPBS weights the particles with all three observations at once.

This notebook is a **sensitivity experiment**, not a calibration: we want to
see what the observations can tell about the parameters, so the priors are as
wide as possible. And it is far more expensive than the previous notebooks
(about 15 000 dynamical runs, a few minutes on 6 processes), so the particles
run in parallel.

AdaPBS is the work of the MuSA authors. If you use it, please cite Aalstad et
al. (2026, GMD, https://doi.org/10.5194/gmd-19-8565-2026) and Alonso-González
et al. (2022, GMD, https://doi.org/10.5194/gmd-15-9127-2022).

In [ ]:
import time

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from oggm import cfg, utils, workflow, tasks

from oggm_adapbs import glogit, gexpit, weighted_summary, weighted_quantile
from oggm_adapbs.dynamic import (dynamic_calibration_adapbs, dynamic_prior,
                                 run_dynamic_particle, iceboost_volume,
                                 terminus_step_km2, area_error_km2)

# Plot colours (prior in grey, observations and OGGM's workflow in black)
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
GREY = '#999999'

## Glacier and data

Vernagtferner (Ötztal Alps, Austria) again, from the OGGM preprocessed
directories at level 3 (W5E5 climate, elevation band flowlines). Level 3
contains the inversion flowlines, and the MB calibration and inversion of
OGGM's standard workflow (which we use for comparison only).

The climate file of these directories starts in 1901: W5E5 itself covers
1979-2019, and is extended back with GSWP3-W5E5. We need it: our historical runs
start 30 years before the RGI date.

In [ ]:
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir('adapbs_dynamic_calibration', reset=True)

rgi_id = 'RGI60-11.00719'  # Vernagtferner
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2025.6/elev_bands/W5E5/per_glacier/')
gdir = workflow.init_glacier_directories([rgi_id], from_prepro_level=3,
                                         prepro_border=80,
                                         prepro_base_url=base_url)[0]

# The temperature bias file of OGGM's informed calibration (centre of the temp_bias prior)
temp_bias_file = ('https://cluster.klima.uni-bremen.de/~oggm/ref_mb_params/oggm_v1.6/'
                  'w5e5_rgi6_perglacier_temp_bias_v2025.6.2.csv')

with xr.open_dataset(gdir.get_filepath('climate_historical')) as clim:
    years = clim.time.dt.year.values
print(f'RGI date: {gdir.rgi_date}, RGI area: {gdir.rgi_area_km2:.2f} km2, '
      f'climate data: {years[0]}-{years[-1]}')

# The three observations

**Geodetic MB.** The mean MB 2000-2019 (Hugonnet et al., 2021), per unit RGI
area, as in the previous notebooks. Its error includes errors which are shared
by many glaciers, and is large for a single glacier. We use 0.2 times this
error (`mb_err_factor`): a choice, which says that we trust the geodetic MB
more than the other observations.

**Area.** The RGI area, at the RGI date. To be precise, the model area at the
start of the year after the RGI date (`state_year`, the end of the RGI year),
and only where the ice is thicker than 2 m (OGGM's `area_min_h`, so that a thin
layer of ice in front of the glacier does not count). Its error has two terms:

$$\sigma_A^2 = (0.05 \, A_{RGI})^2 + 0.25 \, \Delta A^2$$

- 5% of the area, for errors in the outline and its date;
- a discretisation term: the model area changes one grid point at a time, so it
  cannot match the observation more closely than the area of one grid point at
  the terminus ($\Delta A$, the largest of the last 5 points of the main
  flowline: `terminus_step_km2`). With the factor 0.25, this term is half a
  grid point. It matters for small glaciers only.

**Mean ice thickness.** The IceBoost v2 volume divided by its area, against the
model volume divided by the model area (all the ice), at the same date as the
area. Why the mean thickness and not the volume?

- the volume grows with the glacier size, the mean thickness much less: a
  relative error means about the same for all glaciers;
- in the model, the volume and the area are not independent: a glacier which
  is too large has too much volume as well, and would be penalised twice.

IceBoost gives an error for each glacier (`iceboost_volume` reads it), but it
is large: we use 20% of the mean thickness instead (`volume_rel_err=0.2`, the
default).

In [ ]:
state_year = gdir.rgi_date + 1
period = gdir.settings['geodetic_mb_period']
geod = utils.get_geodetic_mb_dataframe().loc[[rgi_id]].set_index('period').loc[period]
ref_mb, ref_mb_err = geod['dmdtda'] * 1000, geod['err_dmdtda'] * 1000  # kg m-2 yr-1

area_step = terminus_step_km2(gdir.read_store('model_flowlines'))
area_sigma = area_error_km2(gdir.rgi_area_km2, area_step)

vol, vol_err, vol_area = iceboost_volume(gdir, return_area=True)
thick = vol / vol_area * 1e3  # km3 / km2 -> m

targets = pd.DataFrame({
    'observation': [ref_mb, gdir.rgi_area_km2, thick],
    'error (data)': [ref_mb_err, np.nan, vol_err / vol * thick],
    'sigma (AdaPBS)': [0.2 * ref_mb_err, area_sigma, 0.2 * thick],
}, index=['MB 2000-2019 (kg m-2 yr-1)', f'area {state_year} (km2)',
          f'mean thickness {state_year} (m)'])
print(f'Terminus grid point: {area_step:.3f} km2. '
      f'IceBoost volume: {vol:.3f} ± {vol_err:.3f} km3 ({vol_err / vol:.0%})')
targets.round(3)

- The geodetic MB: -913 ± 154 kg m-2 yr-1, used with an error of 31.
- The area: 8.56 km². The terminus grid point is 0.171 km² (2% of the area),
  so that the discretisation term barely adds anything to the 5%: sigma
  0.436 km² instead of 0.428.
- The mean thickness: 51.6 m (0.441 km³ over 8.56 km²). The IceBoost error is
  66% (34 m): anything between 17 and 86 m would be within one sigma, and the
  observation would say almost nothing. With 20%, the error is 10.3 m.

# The forward model

`run_dynamic_particle` runs the model for one parameter set. It does three
things:

1. **Inversion.** The thickness of the glacier at the RGI date, with OGGM's
   inversion tasks (`apparent_mb_from_any_mb`, `prepare_for_inversion`,
   `mass_conservation_inversion`, `filter_inversion_output`,
   `init_present_time_glacier`). The inversion assumes a glacier in
   equilibrium: the MB parameters set the shape of the MB profile (its offset
   is removed), and therefore the ice flux, and A the ice creep. The softer the
   ice (the larger A), the thinner the ice which carries the same flux: the
   thickness scales with $A^{-1/5}$. The parameter is the log10 of a factor on
   OGGM's default A (`log10_glen_a_factor`: 0 is the default A, 1 is ten times
   A). We do not use the A of the regional calibration: this would bring the
   IceBoost volume of the region into the prior.
2. **Growth.** The inverted glacier is the glacier *at the RGI date*. To get a
   glacier 30 years earlier, we run the model for 20 years with a constant
   climate (the mean climate of the 30 years before the RGI date) and the
   particle's temperature bias plus an offset (`growth_temp_offset`). A negative
   offset makes the glacier grow. This is OGGM's dynamic spinup, except that
   OGGM finds the temperature bias by minimisation (and runs the growth period
   for as long as the spinup period). The growth period has no calendar date:
   it only produces a starting glacier.
3. **Historical run** from 30 years before the RGI date (here 1974) to the end
   of the geodetic period (2020), with the historical climate and the
   particle's temperature bias (without the offset).

The particle writes its settings and its inversion to the glacier directory
(with its own file suffix, so that several particles can run at the same time)
and deletes them at the end. It returns the three predictions, the inverted
volume, the volume after the growth period and the time series of the volume,
area and mass, including the growth period.

Let us run a few particles by hand. We start from the MB parameters of OGGM's
static calibration (stored in the preprocessed directory) and the default A,
then change the growth offset and A.

In [ ]:
mb_params = ['melt_f', 'prcp_fac', 'temp_bias']
calib = {p: gdir.settings[p] for p in mb_params}  # OGGM's static calibration
print('OGGM static calibration:', {p: round(v, 2) for p, v in calib.items()})

run_kwargs = dict(state_year=state_year, spinup_years=30, growth_years=20,
                  ref_period=(2000, 2020), glen_a_ref=cfg.PARAMS['inversion_glen_a'])
demo = {'OGGM calibration, default A': {},
        'growth offset -1 K': {'growth_temp_offset': -1},
        'growth offset +1 K': {'growth_temp_offset': 1},
        'A x 5': {'log10_glen_a_factor': np.log10(5)}}
base = {**calib, 'log10_glen_a_factor': 0., 'growth_temp_offset': 0.}
runs = {name: run_dynamic_particle(gdir, {**base, **change}, '_demo', **run_kwargs)
        for name, change in demo.items()}

pd.DataFrame({name: {'inverted volume (km3)': r['volume_inversion_km3'],
                     'volume after growth (km3)': r['volume_growth_km3'],
                     'MB 2000-2019 (kg m-2 yr-1)': r['pred'][0],
                     f'area {state_year} (km2)': r['pred'][1],
                     f'mean thickness {state_year} (m)': r['pred'][3],
                     'run time (s)': r['run_time']}
              for name, r in runs.items()}).T.round(3)

A plot of the volume, area and MB of a run against the observations (we reuse
it below). The volume panel shows the IceBoost volume with the error used by
AdaPBS (20%, in black) and the IceBoost error (grey); AdaPBS uses the mean
thickness, which is the volume divided by the area. The MB panel shows the
annual specific MB (the mass change of each year, per unit RGI area, as the
geodetic MB) and the geodetic MB 2000-2019 with its error. The grey area is the
growth period.

In [ ]:
def specific_mb(mass):
    # Annual specific MB (kg m-2 yr-1, per unit RGI area) from the mass series
    return np.diff(mass, axis=-1) / gdir.rgi_area_m2


def plot_obs(axes, growth_years=20):
    ys = state_year - 30
    for ax in axes:
        if growth_years:
            ax.axvspan(ys - growth_years, ys, color='0.92')
        ax.set_xlabel('Year')
    axes[0].errorbar(state_year, vol, yerr=vol_err, fmt='none', color=GREY, capsize=3,
                     label='IceBoost (its error)')
    axes[0].errorbar(state_year, vol, yerr=0.2 * vol, fmt='o', color='k', capsize=3,
                     label='IceBoost (20%)')
    axes[1].errorbar(state_year, gdir.rgi_area_km2, yerr=area_sigma, fmt='o',
                     color='k', capsize=3, label='RGI')
    s = 0.2 * ref_mb_err
    axes[2].fill_between([2000, 2020], ref_mb - s, ref_mb + s, color='k', alpha=0.2)
    axes[2].plot([2000, 2020], [ref_mb, ref_mb], color='k', label='Geodetic MB')
    axes[0].set_ylabel('Volume (km$^3$)')
    axes[1].set_ylabel('Area, ice thicker than 2 m (km$^2$)')
    axes[2].set_ylabel('Specific MB (kg m$^{-2}$ yr$^{-1}$)')


def plot_run(axes, t, volume, area, mass, **kwargs):
    # One run: lines
    axes[0].plot(t, volume, **kwargs)
    axes[1].plot(t, area, **kwargs)
    axes[2].plot(t[:-1], specific_mb(mass), **kwargs)


def plot_band(axes, ds, color=C[0], label='AdaPBS', fill=True):
    # The particles of an AdaPBS run: weighted median and 5-95% range
    t = ds.time.values
    w = ds.weight.values
    series = [(t, ds.volume_km3.values), (t, ds.area_km2.values),
              (t[:-1], specific_mb(ds.mass_kg.values))]
    for ax, (tt, x) in zip(axes, series):
        ok = np.all(np.isfinite(x), axis=1)
        q = np.array([weighted_quantile(x[ok, i], w[ok], [0.05, 0.5, 0.95])
                      for i in range(x.shape[1])])
        if fill:
            ax.fill_between(tt, q[:, 0], q[:, 2], color=color, alpha=0.25, lw=0)
        else:
            ax.plot(tt, q[:, [0, 2]], color=color, lw=1, ls=':')
        ax.plot(tt, q[:, 1], color=color, label=f'{label} (median, 5-95%)')


t_demo = np.arange(state_year - 30 - 20, 2021)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_obs(axes)
for (name, r), c in zip(runs.items(), C):
    plot_run(axes, t_demo, r['volume_km3'], r['area_km2'], r['mass_kg'], color=c, label=name)
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()

- With OGGM's static calibration and the default A (blue), the inverted volume
  is 0.587 km³ (61 m mean thickness in 2004, more than IceBoost: the regional A of
  the preprocessed directory is 2.4 times larger, which gives thinner ice). The glacier shrinks a little
  during the growth period. Its MB 2000-2019 is -971 kg m-2 yr-1, not -913: the
  static calibration does not hold for the dynamical run. This is the mismatch
  which OGGM's dynamic calibration corrects.
- The growth offset sets the starting glacier: 0.702 km³ after a growth period
  1 K colder, 0.389 km³ 1 K warmer. The differences persist until 2020. With
  +1 K, the glacier tongue thins until it disappears after 1990 (the area drops
  to 6.4 km² in 2020), and the MB 2000-2019 is less negative (-728 against
  -1028 with -1 K): a smaller glacier has less ice at low elevations.
- Five times the default A gives 27% less ice ($5^{-1/5} = 0.72$): a mean
  thickness of 43 m instead of 61 m, with the same area and MB.
- The MB during the growth period is constant (constant climate). The MB of the
  historical run follows the climate: it is strongly positive at the end of the
  1970s (about +1000 kg m-2 yr-1 in 1978-1979). Whether the model gets these
  years right is not tested by any of our observations, which are all after
  2000; but the 30-year spinup goes through them.

Each run takes about 0.1 s.

# The prior

This is a sensitivity experiment: we want to know what the observations can
constrain, so the priors are wide. For each parameter, a logit-normal centred
in the middle of its bounds, with a standard deviation of 1.4 in normal space:
close to uniform between the bounds (see the second notebook).

- `melt_f`, `prcp_fac`: OGGM's bounds (from the glacier settings);
- `temp_bias`: ± 5 K around the value of the temperature bias file (OGGM's
  bounds are ± 15 K, but most of these glaciers would vanish or fill the whole
  domain);
- `log10_glen_a_factor`: ± 1, from 0.1 to 10 times OGGM's default A;
- `growth_temp_offset`: ± 3 K.

`dynamic_prior` returns the default prior of `dynamic_calibration_adapbs`; we
change its bounds and centres.

In [ ]:
tb_centre = dynamic_prior(gdir, temp_bias_file_path=temp_bias_file)['centre']['temp_bias']
prior_bounds = {'melt_f': (gdir.settings['melt_f_min'], gdir.settings['melt_f_max']),
                'prcp_fac': (gdir.settings['prcp_fac_min'], gdir.settings['prcp_fac_max']),
                'temp_bias': (tb_centre - 5, tb_centre + 5),
                'log10_glen_a_factor': (-1, 1),
                'growth_temp_offset': (-3, 3)}
prior_centre = {p: (lo + hi) / 2 for p, (lo, hi) in prior_bounds.items()}
prior_sd = 1.4
prior = dynamic_prior(gdir, prior_centre=prior_centre, prior_bounds=prior_bounds)
params = prior['params']

# Samples of the prior, for the plots and the table
rng = np.random.default_rng(0)
z = (glogit(prior['x_centre'], prior['x_min'], prior['x_max'])
     + prior_sd * rng.standard_normal((20000, len(params))))
prior_samples = gexpit(z, prior['x_min'], prior['x_max'])
prior_q = weighted_summary(prior_samples, np.ones(len(z)) / len(z), params)
pd.concat([pd.DataFrame({'min': prior['x_min'], 'max': prior['x_max']}, index=params),
           prior_q], axis=1).round(2)

# Running AdaPBS

`dynamic_calibration_adapbs` reads the observations, builds the prior and runs
AdaPBS with `run_dynamic_particle` as forward model. Its main arguments:

- `spinup_years=30`, `growth_years=20`: the length of the historical run before
  the RGI date and of the growth period;
- `mb_err_factor=0.2`, `area_err=0.05`, `area_k=0.25`,
  `volume_target='thickness'`, `volume_rel_err=0.2`: the observations and their
  errors described above (these are the defaults);
- `reject_sigma=5`: a particle whose *inverted* glacier is more than 5 errors
  away from the IceBoost thickness is not run further (it gets a zero weight).
  This saves the dynamical runs of hopeless particles. The thickness at the RGI
  date is not the inverted thickness (the glacier changes during the growth
  and historical runs), so the threshold must be generous;
- `n_workers`: the number of processes running the particles in parallel. We
  leave a few cores free. Use 1 (the default) when OGGM's multiprocessing
  already runs several glaciers in parallel.

With wide priors, AdaPBS needs more iterations: we use 500 particles per
iteration and allow up to 20 iterations.

In [ ]:
adapbs_kwargs = dict(prior_centre=prior_centre, prior_bounds=prior_bounds,
                     prior_sd=prior_sd, temp_bias_file_path=temp_bias_file,
                     n_particles=500, max_iterations=20, seed=0, n_workers=6)

t0 = time.time()
ds = dynamic_calibration_adapbs(gdir, spinup_years=30, growth_years=20, **adapbs_kwargs)
print(f'{ds.sizes["particle"]} runs in {time.time() - t0:.0f} s')
ds

The Dataset has the same structure as in the previous notebooks, with the
predictions of the three observations in `pred` (dimension `obs`), and more
per-particle output: the Glen A, the inverted volume, the volume after the
growth period, the volume, area and mass time series (dimension `time`,
starting with the growth period), the `status` of the run (`ok`, `rejected`, or
the error) and its run time.

The diagnostics per iteration, and the status of the runs:

In [ ]:
def diagnostics(ds):
    d = ds[[v for v in ds.data_vars if v.startswith('adapbs_')]].to_dataframe()
    d.columns = [c.replace('adapbs_', '') for c in d.columns]
    return d

def status(ds):
    s = pd.Series(ds.status.values)
    s[~s.isin(['ok', 'rejected'])] = 'error'
    return s.value_counts()

display(diagnostics(ds).round(3))
status(ds)

AdaPBS needed 19 iterations (9500 runs, about 4.5 minutes on 6 processes),
much more than with informed priors: the wide prior is far from the
posterior. The effective sample size does not increase at every iteration
(AdaPBS recomputes the weights of all the particles so far), and reaches 31% of
500 after the last one.

Half of the particles of the first iteration (250 of 500) gave no prediction:
their glacier outgrew the domain (`Glacier exceeds domain boundaries`) or their
inverted glacier was too far from the observed thickness (`rejected`). They get
a zero weight. After a few iterations, AdaPBS no longer proposes such parameter
sets.

# The posterior

In [ ]:
def particles(ds):
    # All the particles (n, n_params) and their weights
    p = ds.attrs['calibrate_params']
    return np.stack([ds[k].values for k in p], axis=1), ds.weight.values

theta, w = particles(ds)
summary = pd.concat({'prior': prior_q, 'posterior': weighted_summary(theta, w, params)},
                    axis=1)
summary.round(2)

In [ ]:
labels = {'melt_f': 'melt_f (kg m$^{-2}$ day$^{-1}$ K$^{-1}$)', 'prcp_fac': 'prcp_fac',
          'temp_bias': 'temp_bias (K)', 'log10_glen_a_factor': 'log10 of the A factor',
          'growth_temp_offset': 'growth_temp_offset (K)'}
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for k, (p, ax) in enumerate(zip(params, axes)):
    bins = np.linspace(prior['x_min'][k], prior['x_max'][k], 31)
    ax.hist(prior_samples[:, k], bins=bins, density=True, color=GREY, alpha=0.5,
            label='prior')
    ax.hist(theta[:, k], bins=bins, weights=w, density=True, histtype='step',
            color=C[0], lw=2, label='posterior')
    ax.set_xlabel(labels[p])
    ax.set_yticks([])
axes[0].legend(fontsize=9)
plt.tight_layout()

- `melt_f` and `prcp_fac`: low values are favoured (medians 4.69 and 2.14, the
  prior medians are 9.24 and 5.06), but the ranges stay wide. The equifinality
  of the second notebook is still there.
- `temp_bias`: hardly changed.
- Glen's A: the posterior is almost the prior (5-95%: -0.73 to 0.79, prior
  -0.82 to 0.81). The mean thickness, even with an error of 20%, does not
  constrain A here. We see why below.
- The growth offset: offsets warmer than about +1.3 K are ruled out (the
  glacier shrinks too much), colder offsets are not.

In [ ]:
# Equally weighted particles, resampled with the posterior weights
idx = np.random.default_rng(1).choice(len(w), 2000, p=w)
post = ds.isel(particle=idx)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
panels = [(post['melt_f'], post['prcp_fac'], post['temp_bias'],
           labels['melt_f'], labels['prcp_fac'], labels['temp_bias']),
          (post['log10_glen_a_factor'], post['volume_inversion_km3'], post['melt_f'],
           labels['log10_glen_a_factor'], 'inverted volume (km$^3$)', labels['melt_f']),
          (post['log10_glen_a_factor'], post['growth_temp_offset'], post['pred'][:, 2],
           labels['log10_glen_a_factor'], labels['growth_temp_offset'],
           f'mean thickness {state_year} (m)')]
for ax, (x, y, c, xl, yl, cl) in zip(axes, panels):
    sc = ax.scatter(x, y, c=c, s=4, cmap='viridis')
    plt.colorbar(sc, ax=ax, label=cl)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
plt.tight_layout()

- Left: the equifinality of the MB parameters, as in the second notebook.
- Middle: A sets the inverted volume, from 0.35 to 0.88 km³. The MB parameters
  feed into the inversion as well: for the same A, a higher `melt_f` (yellow)
  means a larger mass turnover, a larger ice flux, and thicker ice. But A
  dominates: thickness scales with the flux to the power 1/5.
- Right: why A is not constrained. With stiff ice (low A), the inverted glacier
  is thick, and only particles with a warm growth period, which make the glacier
  thinner, reach the observed thickness in 2004. With soft ice, the inverted
  glacier is thin, and the growth offset can be lower. The growth period
  compensates A.

# The fit to the observations

The predictions of the particles, before (the first iteration, which samples the
prior) and after calibration (weighted), with the observations and the errors
used by AdaPBS:

In [ ]:
obs_labels = ['MB 2000-2019 (kg m$^{-2}$ yr$^{-1}$)', f'Area {state_year} (km$^2$)',
              f'Mean thickness {state_year} (m)']

def pred_summary(ds):
    ok = np.isfinite(ds.pred[:, 0]).values
    q = [weighted_quantile(ds.pred[ok, k].values, ds.weight.values[ok], [0.05, 0.5, 0.95])
         for k in range(3)]
    return pd.DataFrame(q, index=['MB', 'area', 'thickness'], columns=['q5', 'q50', 'q95'])

first = (ds.iteration == 1).values
ok = np.isfinite(ds.pred[:, 0]).values
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for k, ax in enumerate(axes):
    lo, hi = np.quantile(ds.pred[first & ok, k], [0.02, 0.98])
    bins = np.linspace(lo, hi, 50)
    ax.hist(ds.pred[first & ok, k], bins=bins, density=True, color=GREY, alpha=0.5,
            label='prior (iteration 1)')
    ax.hist(ds.pred[ok, k], bins=bins, weights=w[ok], density=True, histtype='step',
            color=C[0], lw=2, label='posterior')
    o, s = float(ds.ref_obs[k]), float(ds.ref_obs_err[k])
    ax.axvline(o, color='k', label='observation')
    ax.axvspan(o - s, o + s, color='k', alpha=0.1)
    ax.set_xlabel(obs_labels[k])
    ax.set_yticks([])
axes[0].legend(fontsize=9)
plt.tight_layout()

pd.concat({'observation': pd.DataFrame({'value': ds.ref_obs.values,
                                        'sigma': ds.ref_obs_err.values},
                                       index=['MB', 'area', 'thickness']),
           'posterior': pred_summary(ds)}, axis=1).round(3)

The particles fit the three observations: the MB (-909 kg m-2 yr-1, 5-95%
-962 to -859), the area (8.41 km², 8.04 to 9.00, slightly below the RGI area)
and the mean thickness (52.8 m, 38 to 68). Before calibration, the area ranged from 0
(glaciers which disappeared) to about 13 km², and the mean thickness from about
10 to 200 m.

# Glacier evolution, and OGGM's standard workflow

How do the posterior glaciers evolve? For comparison, we run OGGM's standard
workflow on the same glacier: the static calibration and the inversion (with
the regional A) of the preprocessed directory, then OGGM's dynamic calibration
(`run_dynamic_melt_f_calibration`), with the same spinup start (1974), the same
area definition (ice thicker than 2 m) and the same MB error (0.2 times the
Hugonnet error). It changes the glacier directory (settings, inversion), so we
run it on a copy.

In [ ]:
ref = tasks.copy_to_basedir(gdir, base_dir=utils.gettempdir('adapbs_dynamic_ref', reset=True),
                            setup='all')
ref.settings['min_ice_thick_for_area'] = 2.
tasks.run_dynamic_melt_f_calibration(ref, ys=state_year - 30, ye=2020,
                                     ref_mb_err_scaling_factor=0.2,
                                     output_filesuffix='_dyncal',
                                     kwargs_run_function={'min_ice_thickness': 2.},
                                     kwargs_fallback_function={'min_ice_thickness': 2.})
with xr.open_dataset(ref.get_filepath('model_diagnostics', filesuffix='_dyncal')) as d:
    oggm_run = d.load()
oggm_run['time'] = np.round(oggm_run.time).astype(int)

diag = ref.get_diagnostics()
oggm = {'melt_f': ref.settings['melt_f'], 'prcp_fac': ref.settings['prcp_fac'],
        'temp_bias': ref.settings['temp_bias'],
        'log10_glen_a_factor': np.log10(ref.settings['inversion_glen_a'] /
                                        cfg.PARAMS['inversion_glen_a']),
        'spinup temp_bias': diag['temp_bias_dynamic_spinup']}
mass = oggm_run.mass_kg
oggm_pred = {'MB': float(mass.sel(time=2020) - mass.sel(time=2000)) / gdir.rgi_area_m2 / 20,
             'area': float(oggm_run.area_min_h_m2.sel(time=state_year)) * 1e-6,
             'thickness': float(oggm_run.volume_m3.sel(time=state_year) /
                                oggm_run.area_m2.sel(time=state_year))}
print(diag['used_spinup_option'])
display(pd.Series(oggm).round(3).to_frame('OGGM'))
pd.concat([pd.Series(oggm_pred).to_frame('OGGM'), pred_summary(ds)], axis=1).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_obs(axes)
plot_band(axes, ds)
plot_run(axes, oggm_run.time.values, oggm_run.volume_m3.values * 1e-9,
         oggm_run.area_min_h_m2.values * 1e-6, oggm_run.mass_kg.values,
         color='k', ls='--', label='OGGM dynamic calibration')
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()

OGGM's dynamic calibration changed `melt_f` from 5.33 (static) to 5.16, and
found a spinup temperature bias of 1.64 K (0.07 K below `temp_bias`, but for a
growth period of 30 years). Its glacier fits the three observations (MB -905
kg m-2 yr-1, area 8.55 km², mean thickness 53.7 m), and lies within the AdaPBS
ranges: it is one of the solutions.

AdaPBS adds the spread:

- volume: the median is close to OGGM's after 1980, but in 1974 the particles
  start anywhere between 0.40 and 0.65 km³;
- area: the median is slightly below OGGM's (and the RGI area), and the range
  widens after 1990: some particles lose their tongue, as the +1 K run above;
- MB: all particles follow the same climate, but the amplitude of the
  year-to-year changes varies, probably because particles with a high
  `melt_f` are more sensitive to temperature. Their mean over 2000-2019 is
  the same.

# Without the growth period

The growth period is the most uncertain part of the workflow. What happens
without it? With `growth_years=0`, the historical run starts in 1974 from the
glacier inverted at the RGI date, and `growth_temp_offset` is not calibrated
(four parameters).

In [ ]:
t0 = time.time()
ds_ng = dynamic_calibration_adapbs(gdir, spinup_years=30, growth_years=0,
                                   calibrate_params=params[:4], **adapbs_kwargs)
print(f'{ds_ng.sizes["particle"]} runs in {time.time() - t0:.0f} s, '
      f'{len(ds_ng.adapbs_iteration)} iterations')
theta_ng, w_ng = particles(ds_ng)
pd.concat({'prior': prior_q,
           'with growth': summary['posterior'],
           'without growth': weighted_summary(theta_ng, w_ng, params[:4])},
          axis=1).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for k, (p, ax) in enumerate(zip(params[:4], axes)):
    bins = np.linspace(prior['x_min'][k], prior['x_max'][k], 31)
    ax.hist(prior_samples[:, k], bins=bins, density=True, color=GREY, alpha=0.5,
            label='prior')
    for d, c, lab in [(ds, C[0], 'with growth'), (ds_ng, C[1], 'without growth')]:
        ax.hist(d[p].values, bins=bins, weights=d.weight.values, density=True,
                histtype='step', color=c, lw=2, label=lab)
    ax.set_xlabel(labels[p])
    ax.set_yticks([])
axes[0].legend(fontsize=9)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_obs(axes)
plot_band(axes, ds, label='with growth')
plot_band(axes, ds_ng, color=C[1], label='without growth', fill=False)
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()

pd.DataFrame({'with growth': [len(ds.adapbs_iteration), ds.attrs['log_evidence']],
              'without growth': [len(ds_ng.adapbs_iteration), ds_ng.attrs['log_evidence']]},
             index=['iterations', 'log evidence']).round(2)

Without the growth period:

- AdaPBS converges faster: 10 iterations (5000 runs, about 2 minutes);
- the MB parameters do not change;
- Glen's A is constrained: values below about half the default A are ruled out
  (5-95%: -0.28 to 0.90), and the median is 2.3 times the default A (0.37). As
  a curiosity, OGGM's workflow uses 2.3 times the default A for this glacier
  (0.37, see the table above). Without the growth offset, nothing compensates A: the
  thickness in 2004 depends directly on the inverted thickness;
- the area stays closer to the RGI area (median 8.55 km²): the historical run
  starts from the RGI outline, and the model area of this glacier changes little
  in 30 years.

The log evidence is higher without the growth period (-14.22 against
-15.06): the observations prefer the simpler model, by a factor of
$e^{0.84} \approx 2.3$. This is weak (and our evidence estimates, with ~150
effective particles, are not precise), but there is no sign that the
observations need the growth period here. This is not surprising: with one
area at one date, nothing tells how different the glacier was in 1974.

# Summary

- The workflow works: one AdaPBS run replaces the static MB calibration, the
  regional inversion and the dynamic calibration. On a glacier of 8.6 km², with
  wide priors, it takes 9500 dynamical runs (4.5 minutes on 6 processes) with
  the growth period, and 5000 without.
- The three observations are fitted, and the result of OGGM's dynamic
  calibration is one of the posterior solutions.
- The MB parameters remain equifinal, as with the geodetic MB alone.
- Glen's A is constrained by the mean thickness (with a 20% error) only
  without the growth period: with it, the growth offset compensates A.
- The observations do not ask for the growth period (log evidence), and
  constrain it only weakly. On this glacier, dropping it costs nothing and makes
  A identifiable.

Open questions:

- The spinup runs through 1974-2000, when the MB of the model is strongly
  positive at times, and none of the observations is before 2000. Observations
  of the past (older outlines, geodetic MB before 2000) would test it.
- Other glaciers: larger ones (Aletsch: much longer runs) and many glaciers
  run on the cluster (`n_workers=1` and OGGM's multiprocessing over glaciers,
  or one pool per glacier).
- Convergence: with about 150 effective particles, the posterior histograms are
  noisy. More particles, several seeds, or a higher `neff_threshold`.
- The errors: 0.2 times the geodetic MB error and 20% of the mean thickness are
  choices.